<a href="https://colab.research.google.com/github/clifcovickh/BiLSTMvsBiLSTMSentiment/blob/main/3_Live_Model_BiLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance deep-translator transformers tensorflow joblib beautifulsoup4

import yfinance as yf
import pandas as pd
import numpy as np
import joblib
import torch
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from deep_translator import GoogleTranslator
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tensorflow.keras.models import load_model

# Load the "Brain" and the "Scalers" we created in Notebook 2
model = load_model('BBCA_BiLSTM_Sentiment_Model.keras')
scaler_features = joblib.load('scaler_features.pkl')
scaler_target = joblib.load('scaler_target.pkl')

# Load FinBERT for sentiment scoring
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

print("Live Analyst environment initialized.")

In [ ]:
def get_lagged_sentiment(ticker="BBCA"):
    # 1. Scrape News from Yesterday
    yesterday = (datetime.now() - timedelta(1)).strftime('%Y-%m-%d')
    rss_url = f"https://news.google.com/rss/search?q={ticker}+when:1d"

    response = requests.get(rss_url)
    soup = BeautifulSoup(response.content, 'xml')
    items = soup.find_all('item')

    headlines = [item.title.text for item in items]
    if not headlines:
        return 0.0 # Neutral if no news found

    # 2. Translate Indonesian -> English
    translator = GoogleTranslator(source='id', target='en')
    translated_headlines = [translator.translate(h) for h in headlines[:10]] # Top 10

    # 3. Score with FinBERT
    inputs = tokenizer(translated_headlines, padding=True, truncation=True, return_tensors="pt")
    outputs = finbert(**inputs)
    predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

    # FinBERT labels: 0: Positive, 1: Negative, 2: Neutral
    # We convert to a single score: Positive=1, Neutral=0, Negative=-1
    scores = predictions.detach().numpy()
    sentiment_score = np.mean(scores[:, 0] - scores[:, 1])

    return sentiment_score

print("Sentiment Engine ready.")

In [ ]:
# 1. Fetch Today's Stock Data
bbca = yf.Ticker("BBCA.JK")
today_data = bbca.history(period="1d")

if today_data.empty:
    print("Market is closed or data not available yet.")
else:
    today_price = {
        'Date': today_data.index[0].strftime('%d/%m/%Y'),
        'Open': today_data['Open'].values[0],
        'High': today_data['High'].values[0],
        'Low': today_data['Low'].values[0],
        'Close': today_data['Close'].values[0],
        'Volume': today_data['Volume'].values[0]
    }

    # 2. Get Lagged Sentiment (Yesterday's News)
    sentiment = get_lagged_sentiment("BBCA")
    today_price['Sentiment_Score'] = sentiment

    # 3. Update the Master CSV
    master_df = pd.read_csv('BBCA_Master_Dataset_BiLSTM.csv')
    new_row = pd.DataFrame([today_price])

    # Prevent duplicate dates
    if today_price['Date'] not in master_df['Date'].values:
        master_df = pd.concat([master_df, new_row], ignore_index=True)
        master_df.to_csv('BBCA_Master_Dataset_BiLSTM.csv', index=False)
        print(f"Successfully added data for {today_price['Date']}")
    else:
        print("Data for today already exists in CSV.")

In [ ]:
# 1. Prepare the 10-day window
recent_data = master_df.tail(10).copy()
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'Sentiment_Score']

# 2. Scale and Reshape
scaled_input = scaler_features.transform(recent_data[features])
scaled_input = scaled_input.reshape((1, 10, 6)) # (Batch, Window, Features)

# 3. Predict
predicted_scaled = model.predict(scaled_input)
predicted_price = scaler_target.inverse_transform(predicted_scaled)[0][0]

# 4. Generate Signal
current_price = recent_data['Close'].values[-1]
difference = predicted_price - current_price
verdict = "📈 UPTREND" if difference > 0 else "📉 DOWNTREND"

print(f"\n--- BBCA DAILY ANALYSIS ---")
print(f"Current Price: Rp {current_price:.2f}")
print(f"Predicted Price (Tomorrow): Rp {predicted_price:.2f}")
print(f"Expected Move: Rp {difference:.2f}")
print(f"Market Verdict: {verdict}")